## Hybrid transfer-learning evaluation

Trains the winning hybrid configuration (lr 5e-5, 60 unfrozen layers, dropout (0.3, 0.6)) under 5-fold cross-validation on the evaluation split (seed 123). The evaluation split is held out from hyperparameter selection, so the mean test accuracy and standard deviation reported here estimate generalization to data the configuration was not chosen on. Backbone and classifier-head weights are transferred once from the synthetic-plus-diffusion pretrained model before per-fold fine-tuning begins.

In [2]:
from google.colab import drive
drive.mount('/content/drive/')

import os
import json
import shutil
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

Mounted at /content/drive/
TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# Winning hybrid configuration.
LR = 5e-5
UNFROZEN_LAYERS = 60
DROPOUT_1 = 0.3
DROPOUT_2 = 0.6

# Dots replaced with "p" so paths don't carry unexpected extensions.
CONFIG_NAME = f"winner_lr{LR:.0e}_lay{UNFROZEN_LAYERS}_drop{DROPOUT_1}_{DROPOUT_2}"
CONFIG_NAME = CONFIG_NAME.replace(".", "p")

# Evaluation split (seed 123), held out from hyperparameter selection.
CV_DIR = "/content/drive/MyDrive/authentic_split_cv_eval"
# Synthetic-plus-diffusion pretrained model whose weights seed the hybrid
# model's backbone and classifier head.
MODEL2_PATH = "/content/drive/MyDrive/augmentation_data/tank_classifier_v3.keras"
RESULTS_ROOT = "/content/drive/MyDrive/authentic_training_cv_eval/model3"
CONFIG_RESULTS_DIR = f"{RESULTS_ROOT}/{CONFIG_NAME}"
RESULTS_LOG = f"{CONFIG_RESULTS_DIR}/results.json"
os.makedirs(CONFIG_RESULTS_DIR, exist_ok=True)


print(f"Checking eval split data in: {CV_DIR}")
missing_paths = []
for fold in range(5):
    fold_dir = f"{CV_DIR}/fold_{fold}"
    for split in ["train", "val", "test"]:
        path = f"{fold_dir}/{split}"
        if not os.path.exists(path):
            missing_paths.append(f"fold_{fold}/{split}")

if missing_paths:
    print(f"Missing eval split folders: {missing_paths}")
else:
    print("Eval split verified.")


if os.path.exists(RESULTS_LOG):
    with open(RESULTS_LOG, "r") as f:
        config_results = json.load(f)
    completed_folds = list(config_results.keys())
    if completed_folds:
        print(f"\nResuming completed folds: {completed_folds}")
    else:
        print("\nNo completed folds found; starting from scratch.")
else:
    config_results = {}
    completed_folds = []

print(f"\nStarting evaluation for configuration {CONFIG_NAME}")
print(f"Learning rate: {LR}, unfrozen layers: {UNFROZEN_LAYERS}, dropout: ({DROPOUT_1}, {DROPOUT_2})")

Checking eval split data in: /content/drive/MyDrive/authentic_split_cv_eval
Eval split verified.
Starting evaluation for configuration winner_lr5e-05_lay60_drop0p3_0p6
Learning rate: 5e-05, unfrozen layers: 60, dropout: (0.3, 0.6)


In [5]:
img_size = (682, 1024)
batch_size = 4
num_classes = 3
AUTOTUNE = tf.data.AUTOTUNE



print(f"Loading pretrained weights from: {MODEL2_PATH}")
model2 = keras.models.load_model(MODEL2_PATH)

model2_base = next(
    (layer for layer in model2.layers
     if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower()),
    None,
)
if model2_base is None:
    raise ValueError("Could not find EfficientNetB4 base in pretrained model")

model2_dense = None
model2_output_dense = None
for layer in model2.layers:
    if isinstance(layer, layers.Dense):
        if layer.units == 128:
            model2_dense = layer
        elif layer.units == num_classes:
            model2_output_dense = layer

model2_base_weights = model2_base.get_weights()
model2_dense_weights = model2_dense.get_weights() if model2_dense else None
model2_output_weights = model2_output_dense.get_weights() if model2_output_dense else None

del model2
keras.backend.clear_session()
print("Pretrained weights extracted; original model cleared from memory.")


for FOLD_IDX in range(5):
    fold_key = f"fold_{FOLD_IDX}"
    if fold_key in completed_folds:
        print(f"\nSkipping {fold_key} (already done)")
        continue

    print(f"\nStarting training for {fold_key}")

    dataset_dir = f"{CV_DIR}/fold_{FOLD_IDX}"
    train_dir = f"{dataset_dir}/train"
    val_dir = f"{dataset_dir}/val"
    test_dir = f"{dataset_dir}/test"

    val_ds = keras.utils.image_dataset_from_directory(
        val_dir,
        labels="inferred",
        label_mode="int",
        image_size=img_size,
        batch_size=batch_size,
        shuffle=False,
        crop_to_aspect_ratio=True,
    )
    test_ds = keras.utils.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="int",
        image_size=img_size,
        batch_size=batch_size,
        shuffle=False,
        crop_to_aspect_ratio=True,
    )

    class_names = val_ds.class_names
    val_ds = val_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)

    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomBrightness(0.2),
        layers.RandomContrast(0.2),
    ])

    t90_aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.10, 0.10),
        layers.RandomBrightness(0.3),
        layers.RandomContrast(0.3),
    ])

    def load_class_ds(class_path, label, img_size, augmentation):
        """Load a single-class folder, augment, and return (image, label) pairs."""
        ds = keras.utils.image_dataset_from_directory(
            class_path,
            labels=None,
            image_size=img_size,
            batch_size=None,
            shuffle=True,
            seed=SEED,
            crop_to_aspect_ratio=True,
        )
        return ds.map(lambda img: (augmentation(img, training=True), tf.cast(label, tf.int32)))

    t72_ds = load_class_ds(f"{train_dir}/t72nolabel", 0, img_size, data_augmentation)
    t80_ds = load_class_ds(f"{train_dir}/t80nolabel", 1, img_size, data_augmentation)
    t90_ds = load_class_ds(f"{train_dir}/t90nolabel", 2, img_size, t90_aug)

    train_ds = (
        t72_ds
        .concatenate(t80_ds)
        .concatenate(t90_ds)
        .shuffle(buffer_size=1000, seed=SEED)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

    # weights=None so ImageNet weights don't overwrite the synthetic-plus-diffusion ones loaded immediately below.
    base_model = keras.applications.EfficientNetB4(
        input_shape=img_size + (3,),
        include_top=False,
        weights=None,
    )
    base_model.set_weights(model2_base_weights)

    base_model.trainable = True
    for layer in base_model.layers[:-UNFROZEN_LAYERS]:
        layer.trainable = False

    trainable_count = sum(layer.trainable for layer in base_model.layers)
    print(f"Trainable layers in base_model: {trainable_count}/{len(base_model.layers)}")

    # Dense layers constructed as named references so their pretrained weights can be loaded after the model is built.
    inputs = keras.Input(shape=img_size + (3,))
    x = keras.applications.efficientnet.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT_1)(x)
    dense_layer = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2))
    x = dense_layer(x)
    x = layers.Dropout(DROPOUT_2)(x)
    output_layer = layers.Dense(num_classes, activation="softmax")
    outputs = output_layer(x)

    model = keras.Model(inputs, outputs, name=f"m3_winner_fold{FOLD_IDX}")

    if model2_dense_weights is not None:
        dense_layer.set_weights(model2_dense_weights)
    if model2_output_weights is not None:
        output_layer.set_weights(model2_output_weights)

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"],
    )

    # One-hot labels
    def to_one_hot(image, label):
        return image, tf.one_hot(label, num_classes)

    train_ds_oh = train_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    val_ds_oh = val_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    test_ds_oh = test_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    
    # Class weights
    class_counts = {
        i: len(os.listdir(f"{train_dir}/{cls}")) for i, cls in enumerate(class_names)
    }
    total = sum(class_counts.values())
    n_classes = len(class_counts)
    class_weight = {i: total / (n_classes * count) for i, count in class_counts.items()}
    print(f"Class weights: {class_weight}")

    # Callbacks
    CHECKPOINT_DIR = f"{CONFIG_RESULTS_DIR}/checkpoints_fold{FOLD_IDX}"
    HISTORY_PATH = f"{CONFIG_RESULTS_DIR}/history_fold{FOLD_IDX}.json"
    MODEL_PATH = f"{CONFIG_RESULTS_DIR}/model_fold{FOLD_IDX}.keras"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    class SaveHistoryCallback(keras.callbacks.Callback):
        """Persist the full training history to disk after every epoch.

        Colab sessions are unstable; writing a JSON snapshot after each
        epoch lets a run be inspected even if the session dies.
        """

        def __init__(self, filepath):
            self.filepath = filepath

        def on_epoch_end(self, epoch, logs=None):
            with open(self.filepath, "w") as f:
                json.dump(self.model.history.history, f)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=15,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=3,
            min_lr=1e-7,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=f"{CHECKPOINT_DIR}/epoch_{{epoch:02d}}_val{{val_accuracy:.3f}}.keras",
            monitor="val_accuracy",
            save_best_only=True,
            verbose=0,
        ),
        SaveHistoryCallback(filepath=HISTORY_PATH),
    ]

    #Train
    history = model.fit(
        train_ds_oh,
        validation_data=val_ds_oh,
        epochs=50,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=2,
    )

    # Save, clear, and reload the persisted model before evaluating on the test set.
    #reloading to guaranteee the evaluation uses the exact saved weights and the inference-ready state (e.g., correct batch-norm statistics) that will be used later during analysis. 
    best_val_acc = max(history.history["val_accuracy"])
    epochs_trained = len(history.history["accuracy"])

    model.save(MODEL_PATH)
    del model
    keras.backend.clear_session()
    model = keras.models.load_model(MODEL_PATH)
    test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)

    print(f"\n{fold_key} results:")
    print(f"  Test accuracy:     {test_acc:.4f}")
    print(f"  Best val accuracy: {best_val_acc:.4f}")
    print(f"  Epochs trained:    {epochs_trained}")

    config_results[fold_key] = {
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss),
        "best_val_accuracy": float(best_val_acc),
        "epochs_trained": epochs_trained,
        "lr": LR,
        "unfrozen_layers": UNFROZEN_LAYERS,
        "dropout_1": DROPOUT_1,
        "dropout_2": DROPOUT_2,
    }
    with open(RESULTS_LOG, "w") as f:
        json.dump(config_results, f, indent=2)
    print(f"  Logged to {RESULTS_LOG}")

    # Freeing GPU memory before next fold; otherwise the EfficientNetB4 graphs accumulate and Colab's GPU runs out.
    del model
    del base_model
    keras.backend.clear_session()


print("\nHybrid transfer-learning eval-split evaluation complete")

if len(config_results) == 5:
    test_accs = [config_results[f"fold_{i}"]["test_accuracy"] for i in range(5)]
    print(f"Per-fold test accuracies: {[f'{a:.4f}' for a in test_accs]}")
    print(f"Mean test accuracy: {np.mean(test_accs):.4f}")
    print(f"Std test accuracy:  {np.std(test_accs):.4f}")
else:
    print(f"Completed folds: {sorted(config_results.keys())}")

Loading pretrained weights from: /content/drive/MyDrive/augmentation_data/tank_classifier_v3.keras
Extracted base and dense weights from pretrained model.
Pretrained weights stored and original model cleared from memory.
Starting training: fold_0
Found 171 files belonging to 3 classes.
Found 171 files belonging to 3 classes.
Found 269 files.
Found 142 files.
Found 103 files.
Trainable layers in base_model: 60/474
Transferred Dense(128) weights from pretrained model.
Transferred output Dense(3) weights from pretrained model.
Class weights: {0: 0.6370370370370371, 1: 1.2027972027972027, 2: 1.6699029126213591}
Epoch 1/50
129/129 - 274s - 2s/step - accuracy: 0.4903 - loss: 1.2954 - val_accuracy: 0.6023 - val_loss: 1.0727 - learning_rate: 5.0000e-05
Epoch 2/50
129/129 - 126s - 977ms/step - accuracy: 0.6284 - loss: 0.8746 - val_accuracy: 0.6550 - val_loss: 0.8935 - learning_rate: 5.0000e-05
Epoch 3/50
129/129 - 126s - 977ms/step - accuracy: 0.6887 - loss: 0.7949 - val_accuracy: 0.6959 - val_